# Actividad 3 - Aprendizaje Supervisado y No Supervisado

**Dataset:** eCommerce behavior data from multi category store
**Autor:** Antonio Ramon Valerio Tejada - A01797448
**Herramienta:** PySpark MLlib

## Importaciones

In [ ]:
import os
import findspark
findspark.init()

import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml.classification import RandomForestClassifier

SEED = 42

## Spark Session

In [ ]:
spark = SparkSession.builder.master("local[*]").appName("eCommerce").config("spark.driver.memory", "8g").getOrCreate()

## Cargar Datos

In [ ]:
import kagglehub

dataset_dir = kagglehub.dataset_download("mkechinov/ecommerce-behavior-data-from-multi-category-store")
csv_path = None

for root, dirs, files in os.walk(dataset_dir):
    for file in files:
        if file == "2019-Oct.csv":
            csv_path = os.path.join(root, file)

df_raw = spark.read.csv(csv_path, header=True, inferSchema=True)
print(f"Registros totales: {df_raw.count()}")

## Muestra Base

In [ ]:
df = df_raw.sample(withReplacement=False, fraction=0.01, seed=SEED)
print(f"Muestra: {df.count()} registros")

## Limpieza y Variables Derivadas

In [ ]:
df_clean = (
    df.dropna(subset=["event_time", "event_type", "product_id", "price", "user_id", "user_session"])
    .filter(F.col("price") >= 0)
    .withColumn("hour", F.hour("event_time"))
    .withColumn("day_of_week", F.date_format("event_time", "E"))
    .withColumn("category_level_1", F.when(F.col("category_code").isNull(), "unknown").otherwise(F.split(F.col("category_code"), "\\.").getItem(0)))
    .withColumn("brand", F.when(F.col("brand").isNull(), "unknown").otherwise(F.col("brand")))
)

q25, q50, q75 = df_clean.approxQuantile("price", [0.25, 0.50, 0.75], 0.05)

df_clean = (
    df_clean.withColumn("price_segment", F.when(F.col("price") <= q25, "bajo").when(F.col("price") <= q50, "medio_bajo").when(F.col("price") <= q75, "medio_alto").otherwise("alto"))
    .withColumn("is_purchase", F.when(F.col("event_type") == "purchase", 1.0).otherwise(0.0))
)

print(f"Datos limpios: {df_clean.count()} registros")

## Particionamiento Estratificado

In [ ]:
df_part = df_clean.withColumn("partition_rule", F.concat_ws("_", F.col("event_type"), F.col("price_segment")))

partition_counts = df_part.groupBy("partition_rule").count().orderBy(F.desc("count"))

N_PER_PARTITION = 600
allocation = partition_counts.withColumn("fraccion_muestreo", F.when(F.col("count") <= N_PER_PARTITION, F.lit(1.0)).otherwise(F.lit(N_PER_PARTITION) / F.col("count")))

fractions = {row["partition_rule"]: min(float(row["fraccion_muestreo"]), 1.0) for row in allocation.collect()}

M = df_part.sampleBy("partition_rule", fractions=fractions, seed=SEED)
print(f"Muestra M: {M.count()} registros")

## Preparacion para Modelado

In [ ]:
model_cols = ["price", "hour", "event_type", "category_level_1", "brand", "price_segment", "day_of_week", "is_purchase", "partition_rule"]

M_model = M.select(*model_cols).dropna(subset=["price", "hour", "is_purchase"])

top_brands = [row["brand"] for row in M_model.groupBy("brand").count().orderBy(F.desc("count")).limit(25).collect()]

M_model = M_model.withColumn("brand_limited", F.when(F.col("brand").isin(top_brands), F.col("brand")).otherwise("other"))

print(f"Datos preparados: {M_model.count()} registros")

## Division Train Test

In [ ]:
M_indexed = M_model.withColumn("row_id", F.monotonically_increasing_id())

train = M_indexed.sampleBy("is_purchase", fractions={0.0: 0.80, 1.0: 0.80}, seed=SEED)

train_ids = train.select("row_id").withColumn("in_train", F.lit(1))

test = M_indexed.join(train_ids, on="row_id", how="left").filter(F.col("in_train").isNull()).drop("in_train")

print(f"Train: {train.count()}, Test: {test.count()}")

## Calcular Pesos de Clase

In [ ]:
class_counts = {row["is_purchase"]: row["count"] for row in train.groupBy("is_purchase").count().collect()}

n_train = sum(class_counts.values())

weight_0 = n_train / (2 * class_counts.get(0.0, 1))
weight_1 = n_train / (2 * class_counts.get(1.0, 1))

train = train.withColumn("class_weight", F.when(F.col("is_purchase") == 1.0, F.lit(weight_1)).otherwise(F.lit(weight_0)))

test = test.withColumn("class_weight", F.when(F.col("is_purchase") == 1.0, F.lit(weight_1)).otherwise(F.lit(weight_0)))

print(f"Peso clase 0: {weight_0:.4f}, Peso clase 1: {weight_1:.4f}")

## Random Forest Classifier

In [ ]:
categorical_cols = ["category_level_1", "brand_limited", "price_segment", "day_of_week"]
numeric_cols = ["price", "hour"]

indexers = [StringIndexer(inputCol=col, outputCol=f"{col}_idx", handleInvalid="keep") for col in categorical_cols]

encoder = OneHotEncoder(inputCols=[f"{col}_idx" for col in categorical_cols], outputCols=[f"{col}_ohe" for col in categorical_cols], handleInvalid="keep")

assembler = VectorAssembler(inputCols=numeric_cols + [f"{col}_ohe" for col in categorical_cols], outputCol="features", handleInvalid="keep")

rf = RandomForestClassifier(labelCol="is_purchase", featuresCol="features", weightCol="class_weight", numTrees=80, maxDepth=7, minInstancesPerNode=5, seed=SEED)

rf_pipeline = Pipeline(stages=indexers + [encoder, assembler, rf])

rf_model = rf_pipeline.fit(train)
predictions = rf_model.transform(test)

print("Modelo entrenado y predicciones generadas")

## Evaluacion

In [ ]:
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator

binary_eval = BinaryClassificationEvaluator(labelCol="is_purchase", rawPredictionCol="rawPrediction", metricName="areaUnderROC")
auc = binary_eval.evaluate(predictions)

multi_eval = MulticlassClassificationEvaluator(labelCol="is_purchase", predictionCol="prediction", metricName="accuracy")
accuracy = multi_eval.evaluate(predictions)

print(f"AUC-ROC: {auc:.4f}")
print(f"Accuracy: {accuracy:.4f}")

## Conclusiones

Se implemento un pipeline completo de machine learning usando PySpark que:

1. Carga y prepara datos de eCommerce
2. Realiza limpieza y ingenieria de caracteristicas
3. Genera muestras estratificadas para entrenamiento
4. Entrena un modelo Random Forest para prediccion de compras
5. Evalua el desempeno del modelo

El modelo puede identificar interacciones con alta probabilidad de conversion.